#  Big Data con PySpark — Notebook 3
## Transformaciones: withColumn, when/otherwise, UDFs

---

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType

spark = (
    SparkSession.builder
    .appName("Vuelos_Transformaciones")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

# Leemos el Parquet limpio del notebook anterior
df = spark.read.parquet("/content/drive/MyDrive/Colab Notebooks/vuelos_limpio.parquet")
df.cache()
print(f"Filas: {df.count():,}")
df.printSchema()

Filas: 446,399
root
 |-- vuelo_id: integer (nullable = true)
 |-- aerolinea: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- destino: string (nullable = true)
 |-- pasajeros: integer (nullable = true)
 |-- distancia_km: integer (nullable = true)
 |-- retraso_min: integer (nullable = true)
 |-- estado: string (nullable = true)
 |-- tarifa_usd: double (nullable = true)
 |-- clase: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- hora: integer (nullable = true)



---
## 1. `withColumn()` — Crear o modificar columnas

In [7]:
# ── Operaciones matemáticas ──
df = (
    df
    .withColumn(
        "ingreso_total_usd",
        F.col("tarifa_usd") * F.col("pasajeros")  # multiplicación columna × columna
    )
    .withColumn(
        "tarifa_cop",
        F.round(F.col("tarifa_usd") * 4200, 0)    # round(expr, decimales)
        # F.round() redondea; 0 decimales = entero
    )
    .withColumn(
        "velocidad_kmh",
        F.round(F.col("distancia_km") / 1.5, 1)   # asumimos 1.5h de vuelo promedio
    )
)

df.select("tarifa_usd", "pasajeros", "ingreso_total_usd", "tarifa_cop", "velocidad_kmh").show(5)

+----------+---------+------------------+----------+-------------+
|tarifa_usd|pasajeros| ingreso_total_usd|tarifa_cop|velocidad_kmh|
+----------+---------+------------------+----------+-------------+
|     72.09|      144|10380.960000000001|  302778.0|       1318.0|
|    478.33|      135|64574.549999999996| 2008986.0|        407.3|
|    498.33|      117|          58304.61| 2092986.0|        721.3|
|    247.92|      128|          31733.76| 1041264.0|       1301.3|
|    200.81|      150|           30121.5|  843402.0|       1170.7|
+----------+---------+------------------+----------+-------------+
only showing top 5 rows


---
## 2. `when() / otherwise()` — Lógica condicional

Es el equivalente al `IF / CASE WHEN` de SQL o el `np.where()` de pandas.

In [8]:
# ── Categorizar el retraso ──
df = df.withColumn(
    "categoria_retraso",
    F.when(F.col("retraso_min") == 0, "Sin retraso")          # condición 1
    .when(F.col("retraso_min") <= 30, "Retraso leve")          # condición 2
    .when(F.col("retraso_min") <= 90, "Retraso moderado")      # condición 3
    .otherwise("Retraso severo")                                # si ninguna aplica
)
# F.when(condicion, valor_si_true)
#   .when(condicion2, valor2)    → se evalúan en orden, gana la primera True
#   .otherwise(valor_default)    → si ninguna condición fue True

df.groupBy("categoria_retraso").count().orderBy("count", ascending=False).show()

+-----------------+------+
|categoria_retraso| count|
+-----------------+------+
|      Sin retraso|313734|
|   Retraso severo| 82910|
| Retraso moderado| 33287|
|     Retraso leve| 16468|
+-----------------+------+



In [9]:
# ── Clasificar la ruta por distancia ──
df = df.withColumn(
    "tipo_ruta",
    F.when(F.col("distancia_km") < 300, "Corta")
    .when(F.col("distancia_km") < 800, "Media")
    .otherwise("Larga")
)

# ── Flag binario: ¿fue puntual? ──
df = df.withColumn(
    "es_puntual",
    F.when(F.col("estado") == "A_TIEMPO", 1).otherwise(0)
    # Crear una columna 0/1 es útil para luego calcular tasas con avg()
)

df.select("origen", "destino", "distancia_km", "tipo_ruta", "estado", "es_puntual").show(8)

+------+-------+------------+---------+--------+----------+
|origen|destino|distancia_km|tipo_ruta|  estado|es_puntual|
+------+-------+------------+---------+--------+----------+
|   LET|    PEI|        1977|    Larga|A_TIEMPO|         1|
|   CTG|    PEI|         611|    Media|A_TIEMPO|         1|
|   MDE|    VVC|        1082|    Larga|DEMORADO|         0|
|   BAQ|    VVC|        1952|    Larga|A_TIEMPO|         1|
|   MTR|    BOG|        1756|    Larga|A_TIEMPO|         1|
|   BAQ|    SMR|        1673|    Larga|A_TIEMPO|         1|
|   SMR|    VVC|         212|    Corta|A_TIEMPO|         1|
|   BOG|    BAQ|        1054|    Larga|A_TIEMPO|         1|
+------+-------+------------+---------+--------+----------+
only showing top 8 rows


---
## 3. Funciones de string

In [10]:
df = (
    df
    # Concatenar dos columnas
    .withColumn(
        "ruta",
        F.concat(F.col("origen"), F.lit(" → "), F.col("destino"))
        # F.concat(a, b, c) → une strings sin separador
        # F.lit(" → ")      → valor literal (constante) como columna
    )
    # Convertir a mayúsculas
    .withColumn("clase_upper", F.upper(F.col("clase")))
    # F.upper() / F.lower() → mayúsculas / minúsculas

    # Extraer subcadena
    .withColumn("origen_prefix", F.substring(F.col("origen"), 1, 2))
    # F.substring(col, posicion_inicio, longitud)
    # posición empieza en 1 (no en 0 como Python)

    # Longitud de un string
    .withColumn("len_aerolinea", F.length(F.col("aerolinea")))
)

df.select("ruta", "clase_upper", "origen_prefix", "len_aerolinea").show(5)

+---------+-----------+-------------+-------------+
|     ruta|clase_upper|origen_prefix|len_aerolinea|
+---------+-----------+-------------+-------------+
|LET → PEI|  ECONOMICA|           LE|            5|
|CTG → PEI|  ECONOMICA|           CT|            5|
|MDE → VVC|    PRIMERA|           MD|            5|
|BAQ → VVC|  ECONOMICA|           BA|            7|
|MTR → BOG|    PRIMERA|           MT|            7|
+---------+-----------+-------------+-------------+
only showing top 5 rows


---
## 4. UDFs — User Defined Functions

Cuando necesitas lógica que las funciones de Spark no cubren, puedes escribir tu propia función Python y registrarla como UDF.

>  **Advertencia de rendimiento:** Las UDFs de Python son lentas porque Spark debe serializar/deserializar datos entre la JVM y Python para cada fila. Úsalas solo cuando no existe una función nativa de Spark equivalente.

In [11]:
# ── Definir función Python normal ──
def clasificar_hora(hora):
    """Clasifica la hora del vuelo en franjas del día."""
    if hora is None:
        return "Desconocido"
    if 5 <= hora < 12:
        return "Mañana"
    elif 12 <= hora < 18:
        return "Tarde"
    elif 18 <= hora < 23:
        return "Noche"
    else:
        return "Madrugada"

# ── Registrar como UDF ──
udf_hora = F.udf(clasificar_hora, StringType())
# F.udf(funcion_python, tipo_de_retorno)
# StringType() → la función devuelve un string
# El tipo de retorno es OBLIGATORIO — Spark necesita saber qué tipo esperar

# ── Aplicar la UDF ──
df = df.withColumn("franja_dia", udf_hora(F.col("hora")))
# Se usa igual que cualquier función de Spark

df.groupBy("franja_dia").count().orderBy("franja_dia").show()

+----------+------+
|franja_dia| count|
+----------+------+
| Madrugada|111316|
|    Mañana|130554|
|     Noche| 93051|
|     Tarde|111478|
+----------+------+



In [12]:
# ── UDF con decorador (forma más limpia) ──
from pyspark.sql.functions import udf

@udf(returnType=StringType())
# udf → decorador que convierte la función Python en UDF de Spark automáticamente
# returnType → tipo de dato que devuelve la función
def nivel_ocupacion(pasajeros):
    """Calcula el nivel de ocupación del vuelo."""
    if pasajeros is None:
        return None
    capacidad = 180  # capacidad máxima asumida
    pct = pasajeros / capacidad
    if pct >= 0.90:
        return "Lleno"
    elif pct >= 0.70:
        return "Alto"
    elif pct >= 0.50:
        return "Medio"
    else:
        return "Bajo"

df = df.withColumn("ocupacion", nivel_ocupacion(F.col("pasajeros")))
df.groupBy("ocupacion").count().orderBy("count", ascending=False).show()

+---------+------+
|ocupacion| count|
+---------+------+
|     Bajo|137070|
|     Alto|124031|
|    Medio|123465|
|    Lleno| 61833|
+---------+------+



### UDF vs Funciones nativas de Spark

| | Funciones nativas (`F.when`, `F.col`, etc.) | UDF Python |
|---|---|---|
| **Velocidad** | ⚡ Rápidas (ejecutan en JVM) | 🐢 Lentas (serialización Python) |
| **Optimización** | ✅ Catalyst las optimiza | ❌ Son cajas negras para Spark |
| **Uso** | Para todo lo que esté disponible | Solo para lógica custom compleja |

**Regla de oro:** Si puedes reemplazar una UDF con `when/otherwise` + funciones de `F`, hazlo.

In [13]:
# ── Guardar el DF enriquecido ──
(
    df
    .write
    .mode("overwrite")
    .parquet("/tmp/vuelos_enriquecido.parquet")
)
print(" Datos enriquecidos guardados")
print("Columnas disponibles:", df.columns)

 Datos enriquecidos guardados
Columnas disponibles: ['vuelo_id', 'aerolinea', 'origen', 'destino', 'pasajeros', 'distancia_km', 'retraso_min', 'estado', 'tarifa_usd', 'clase', 'anio', 'mes', 'hora', 'ingreso_total_usd', 'tarifa_cop', 'velocidad_kmh', 'categoria_retraso', 'tipo_ruta', 'es_puntual', 'ruta', 'clase_upper', 'origen_prefix', 'len_aerolinea', 'franja_dia', 'ocupacion']


---
## Resumen del notebook

```
withColumn(nombre, expr)             →  crear/reemplazar columna
F.col("x") * F.col("y")             →  operaciones entre columnas
F.lit(valor)                         →  valor constante como columna
F.round(col, decimales)              →  redondear
F.when(cond, val).otherwise(val)     →  lógica condicional
F.concat(a, b, c)                    →  unir strings
F.upper() / F.lower()                →  mayúsculas / minúsculas
F.substring(col, inicio, largo)      →  subcadena
F.udf(func, TipoRetorno)             →  función Python → UDF Spark
@udf(returnType=Tipo())              →  forma decorador
```
 **Siguiente:** Agregaciones y GroupBy